In [ ]:
# Core data manipulation
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# ML / clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px  # optional for interactive plots


In [ ]:
# Load CSV
df = pd.read_csv("customerdata.csv", parse_dates=['InvoiceDate'])

# Quick look
print(df.head())
print(df.info())

In [ ]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

In [ ]:
# Latest date in dataset (to calculate recency)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency
    'TotalAmount': 'sum'                                      # Monetary
}).reset_index()

rfm.rename(columns={'InvoiceDate':'Recency','InvoiceNo':'Frequency','TotalAmount':'Monetary'}, inplace=True)
print(rfm.head())

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# KMeans clustering
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Segment'] = kmeans.fit_predict(X_scaled)

print(rfm.head())

In [ ]:
# Segment distribution
sns.countplot(x='Segment', data=rfm)
plt.show()

# Monetary contribution per segment
rfm.groupby('Segment')['Monetary'].sum().plot(kind='bar')
plt.show()

In [ ]:
score = silhouette_score(X_scaled, rfm['Segment'])
print(f"Silhouette Score: {score:.2f}")